# 2D FEA Solver – Theory, Validation & Mesh Convergence

M.Tech Mechanical + Python portfolio notebook.

## Contents
1. Truss element stiffness & validation
2. Euler-Bernoulli beam – cantilever analytical comparison
3. CST plane-stress plate – cantilever, stress contour, mesh convergence
4. ML surrogate for max von Mises prediction


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import matplotlib.pyplot as plt
from fea_truss import TrussElement, solve_truss
from fea_beam import BeamElement, solve_beam, cantilever_analytical
from fea_plate import cantilever_plate_mesh, generate_rect_mesh, Material, solve_plate
from ml_surrogate import train_surrogate
print('Imports OK')

## 1. Truss – single bar axial test

Analytical: $u = FL/(EA)$

In [ ]:
nodes = np.array([[0.,0.],[1.,0.]])
elems = [TrussElement(0,1,E=210e9,A=0.001)]
res = solve_truss(nodes, elems, loads={1:(1000.,0.)}, fixed_nodes=[0])
u_fea = res['displacements'][1,0]
u_exact = 1000*1.0/(210e9*0.001)
print(f'FEA u = {u_fea:.6e} m, Exact = {u_exact:.6e} m, error = {abs(u_fea-u_exact)/u_exact*100:.4f}%')

## 2. Beam cantilever – tip load

Analytical tip deflection: $\delta = PL^3/(3EI)$

In [ ]:
L,E,I,P = 1.0,210e9,1e-6,-1000.
u_exact = cantilever_analytical(L,abs(P),E,I)*np.sign(P)
for ne in [1,2,4,8]:
    xs = np.linspace(0,L,ne+1)
    nodes = np.column_stack([xs,np.zeros_like(xs)])
    elems = [BeamElement(i,i+1,E=E,I=I) for i in range(ne)]
    r = solve_beam(nodes,elems,loads={ne:(0.,P,0.)},fixed_dofs=[0,1,2])
    tip = r['displacements'][-1,1]
    print(f'n_el={ne:2d}  tip={tip*1e3:8.4f} mm  error={abs(tip-u_exact)/abs(u_exact)*100:6.2f}%')

## 3. CST Plate – cantilever & mesh convergence

In [ ]:
res = cantilever_plate_mesh(L=1.0,H=0.2,nx=16,ny=4,tip_load=-1000)
print(f'Max von Mises = {res["max_vm"]/1e6:.2f} MPa')
print(f'Max displacement = {res["max_disp"]*1e3:.4f} mm')
print(f'Elements = {len(res["elements"])}, Nodes = {len(res["nodes"])}')

In [ ]:
nx_list=[4,8,12,16,24]
vms, us = [], []
for nx in nx_list:
    r = cantilever_plate_mesh(L=1.,H=0.2,nx=nx,ny=max(2,nx//5),tip_load=-1000)
    vms.append(r['max_vm']/1e6); us.append(r['max_disp']*1e3)
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].plot(nx_list,vms,'o-'); ax[0].set_xlabel('nx'); ax[0].set_ylabel('max σ_vm (MPa)'); ax[0].set_title('Stress convergence')
ax[1].plot(nx_list,us,'s-'); ax[1].set_xlabel('nx'); ax[1].set_ylabel('max |u| (mm)'); ax[1].set_title('Displacement convergence')
plt.tight_layout(); plt.show()

## 4. ML Surrogate (optional – takes ~20-40 s)

In [ ]:
# model, metrics, _ = train_surrogate(n_samples=60, seed=42)
# print(metrics)